In [5]:
# Indian Monsoon Progression - 20% Rainfall Line
# =============================================

# --- 1. Imports ---
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from datetime import datetime

# --- 2. User settings ---
datafile = "/glade/work/rneale/data/GPCP/gpcp.mon.mean.197901-201607.nc"   # Change to your file path
varname  = "precip"            # Variable name in dataset
percent_threshold = 0.20       # 20% threshold
region_extent = [60, 100, 5, 35]  # lon_min, lon_max, lat_min, lat_max

# --- 3. Read in the dataset ---
ds = xr.open_dataset(datafile)
pr = ds[varname]  # Expect dims: time, lat, lon
# Make sure time is decoded as datetime
if not np.issubdtype(pr.time.dtype, np.datetime64):
    pr['time'] = xr.decode_cf(ds).time

display(pr)

# --- 4. Create a day-of-year climatology ---
# Ensure we have no leap days for simplicity
pr = pr.sel(time=~((pr['time'].dt.month == 2) & (pr['time'].dt.day == 29)))

# Group by day of year
pr_clim = pr.groupby('time.dayofyear').mean('time')

# --- 5. Compute cumulative climatology ---
# Sum over days to get cumulative rainfall
cum_pr = pr_clim.cumsum(dim='dayofyear')



# --- 6. Find the first day when cumulative >= threshold ---
threshold_broadcast = threshold_val.broadcast_like(cum_pr)

def day20_func(cum_series, thresh_series):
    arr = cum_series >= thresh_series
    if not np.any(arr):
        return np.nan
    return int(np.argmax(arr) + 1)  # +1 because dayofyear starts at 1

day20 = xr.apply_ufunc(
    np.vectorize(day20_func, otypes=[float]),
    cum_pr,
    threshold_broadcast,
    input_core_dims=[['dayofyear'], ['dayofyear']],
    vectorize=True,
    dask='parallelized'
)

# --- 7. Plot the 20% progression line ---
fig = plt.figure(figsize=(10, 8))
proj = ccrs.PlateCarree()
ax = plt.subplot(1, 1, 1, projection=proj)
ax.set_extent(region_extent, crs=proj)

# Add map features
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.5)

# Plot contours of 20% date
c = ax.contour(
    day20.lon, day20.lat, day20,
    levels=[150, 160, 170],  # Example days (adjust to highlight progression)
    colors='red',
    linewidths=2,
    transform=proj
)

ax.clabel(c, fmt="%d", inline=True, fontsize=10)
ax.set_title("Indian Monsoon 20% Rainfall Progression Line (Climatology)",
             fontsize=14)
plt.show()


ValueError: setting an array element with a sequence.